# Calibrar los umbrales de vigilancia con tus datos

Este notebook **no escribe nada**. Responde una sola pregunta:

> ¿Qué umbral me da la mayor detección sin pasar de X alertas por día?

Cómo lo mide, con tus propias series:

1. **Volumen**: corre el motor sobre tus datos tal cual y cuenta cuántas notificaciones saldrían.
2. **Detección**: mete fallas conocidas en algunas de tus series reales —apagón, escalón a la mitad,
   pico— y mide cuántas de esas el motor sí avisa.

Un umbral alto avisa poco y se pierde cosas; uno bajo avisa todo y nadie lo lee. **La curva la
calcula el notebook; el punto lo elegís vos** con `OBJETIVO_ALERTAS_DIA`.

In [ ]:
# Parámetros
import json, os

FECHA_EJECUCION = os.getenv("VG_FECHA_EJECUCION") or None

# Cuántas alertas por día está dispuesto a leer el que las recibe. Es LA decisión.
OBJETIVO_ALERTAS_DIA = float(os.getenv("VG_OBJETIVO") or 3)

# Umbrales a probar. "alerta" es el desvío robusto desde el que se avisa; ATENCION y CRITICO
# se acomodan solos alrededor. (VG_REJILLA la pisa, en JSON.)
REJILLA = json.loads(os.getenv("VG_REJILLA") or "null") or {
    "alerta": [4.0, 5.0, 6.0, 8.0, 10.0],
    "desvio_relativo_minimo": [0.05, 0.10, 0.20],
}
N_FALLAS = int(os.getenv("VG_N_FALLAS") or 30)   # series reales que se rompen para medir detección
print(f"{len(REJILLA['alerta']) * len(REJILLA['desvio_relativo_minimo'])} configuraciones a probar")

## 1. Leer toda la historia

In [ ]:
import sys, time
sys.path.insert(0, os.getcwd())
import pandas as pd
import vig_oracle as io
from vig_engine import calibrar, inyectar_fallas, bloque_config_vig, Fechas

log = io.configurar_logging("calibracion")
cfg = io.build_config(fecha_ejecucion=FECHA_EJECUCION)
f = Fechas.desde(cfg.fecha_ejecucion)
desde = (f.ayer - pd.Timedelta(days=cfg.dias_historia)).normalize()

with io.conexion_origen() as conn:
    datos = io.leer_fuente(conn, cfg, desde)
{k: len(v) for k, v in datos.items()}

## 2. Probar los umbrales

Cada configuración corre el motor dos veces (con y sin fallas inyectadas), así que esto tarda.
Con muchas series conviene arrancar con una rejilla chica.

In [ ]:
t0 = time.time()
tabla = calibrar(datos, cfg, rejilla=REJILLA, objetivo_alertas_dia=OBJETIVO_ALERTAS_DIA,
                 n_fallas=N_FALLAS)
print(f"{len(tabla)} configuraciones en {time.time() - t0:.0f}s")
tabla

## 3. Cómo leer la tabla

| Columna | Qué mirar |
|---|---|
| `alertas_por_dia` | cuántas notificaciones saldrían por día con ese umbral |
| `deteccion` | qué fracción de las fallas inyectadas avisó |
| `eventos` | cuántas filas van a `VIG_EVENTO` por corrida (aunque no se notifiquen) |
| `recomendado` | la que más detecta sin pasarse de tu objetivo |

Tres situaciones típicas:

- **Ninguna fila baja de tu objetivo**: estás vigilando demasiadas series, o hay un problema real y
  grande. Subí el umbral, subí `desvio_relativo_minimo`, o vigilá a un nivel más agregado.
- **Detección alta con pocas alertas**: perfecto, quedate con esa.
- **Detección baja en todas**: las fallas inyectadas son más chicas que el ruido de tus series. Fijate
  `materialidad_minima` de cada vigilancia, o mirá si conviene el grano semanal en vez del diario.

## 4. El bloque para pegar en `vig_oracle.py`

In [ ]:
elegido = tabla[tabla.recomendado].iloc[0]
print(bloque_config_vig(elegido))

## 5. Después

1. Pegá el bloque en `vig_oracle.py` y corré `run_vigilancia.ipynb` unos días.
2. Que la gente anote en `VIG_CAUSA` qué era cada cosa: eso es lo que después permite afinar por
   métrica, con precisión medida de verdad y no contra fallas simuladas.
3. Volvé a correr esta calibración cada tanto, o cuando cambie el volumen del negocio.